# Deep Learning Project Work

In this project, I will explore the benchmarking of DINO models for an image classification task. The models to be evaluated include:

- (?) DINOv2 (facebook/dinov2) (?)
- DINOv2 base with registers (facebook/dinov2-with-registers-base)
- DINOv3 base (facebook/dinov3-vitb16-pretrain-lvd1689m)

The dataset on which the models will be trained and evaluated is the dataset available at the [link](https://zenodo.org/records/10137731) relative to [this paper](https://onlinelibrary.wiley.com/doi/10.1111/gcb.17078), regarding micrometeorological conditions across 49 wildlife cameras in South Africa's Maloti-Drakensberg and the Swiss Alps, with classifications for **overcast**, **sunshine**, **hail**, and **snow** conditions.

In the original paper, 

In [1]:
import numpy as np
import pandas as pd
import random
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
import ipywidgets as widgets
from ipywidgets import interact

SEED=42

DATASET_PATH = "/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_weather_ct/"

IMAGES_METADATA_CSV = DATASET_PATH + "image_metadata.csv"

IMAGES_DIR = DATASET_PATH + "images/"

CLASS_NAMES = [
    "background",
    "sunshine",
    "snow",
    "hail",
]

def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(SEED)

## Analyzing dataset structure

Attached to the dataset there are CSV files with metadata for images and cameras. The main file for images is `image_data.csv`, with these columns, not all of which are needed/useful for model training:

- `DatasetOutFileName`: image file name.
- `label_sample`: sampling strategy used to select the image (not needed for model training).
- `Region`: camera region (e.g. Maloti-Drakensberg `ZA` or Swiss Alps `CH`).
- `cam`: unique camera ID that captured the image.
- `DateTimeOriginal`: original date/time from image metadata (raw format, may contain colons).
- `DateTime`: reformatted date/time (standardized representation).
- `DOY`: day of year ($1–366$).
- `Hour`: hour of capture ($0–23$).
- `Minute`: minute of capture ($0–59$).
- `Temperature`: ambient temperature recorded by the sensor at capture time.
- `Flash`: whether flash fired ($1$) or not ($0$).
- `ExposureTime`: shutter exposure time in seconds (decimal).
- `weather`: numeric weather class label (target for the model, e.g. 0–3).
- `weather_desc`: textual weather class (e.g. `background`, `sunshine`, `hail`, `snow`).
- `cis_fold`: fold index ($0–4$) for **Cis fivefold cross‑validation**, a within‑domain, image‑level random split where images from all cameras are randomly assigned to folds. Folds are mutually exclusive, stratified by the four study sites, and each fold is iteratively used as validation while the other four form the training set.
- `trans_fold`: fold index ($0–4$) for **Trans fivefold cross‑validation**, a camera‑level split where entire cameras (all their images) are assigned to folds so that no camera appears in both training and validation. Folds are mutually exclusive and stratified by the four study sites.
- `holdout`: boolean indicator (`TRUE` if reserved exclusively for the final test set).
- `cis_prod_fold`: alternative Cis fold variant (also a fivefold, site‑stratified Cis split) used for production/alternate experiment splits or final model selection; similar semantics to `cis_fold` but intended for deployment/production evaluation.


In [2]:
import os
from collections import defaultdict

df_metadata_images = pd.read_csv(IMAGES_METADATA_CSV)
df_metadata_images.head()

,DatasetOutFileName,label_sample,Region,cam,DateTimeOriginal,DateTime,DOY,Hour,Minute,Temperature,Flash,ExposureTime,weather,weather_desc,cis_fold,trans_fold,holdout,cis_prod_fold
0,000001.JPG,systematic,ZA,SA16,2022:01:12 21:01:12,12/01/2022 21:01,12,21,1,10,1,0.065986,0,background,2,4,False,4
1,000002.JPG,strategic,ZA,SA07,2021:12:25 03:03:27,25/12/2021 03:03,359,3,3,2,1,0.065986,3,hail,4,2,False,1
2,000003.JPG,systematic,ZA,SA06,2022:04:01 15:00:27,01/04/2022 15:00,91,15,0,6,0,0.004672,1,sunshine,1,1,False,1
3,000004.JPG,strategic,CH,C02,2021:09:04 01:00:01,04/09/2021 01:00,247,1,0,7,1,0.065986,0,background,6,6,True,5
4,000005.JPG,systematic,CH,C17,2021:08:31 09:01:31,31/08/2021 09:01,243,9,1,5,0,0.002625,1,sunshine,2,3,False,5


## Load dataset

Because the dataset identifies images using only their file names (without paths), and images are organized in subdirectories by class ID, a function is needed to resolve the full image paths based on the `DatasetOutFileName` and the directory structure.

In [3]:
def resolve_filename(fname: str) -> str:
    hits = file_map.get(fname)
    if not hits:
        return None
    if len(hits) > 1:
        # even if should not happen, warn if multiple matches, default to first
        print(f"Warning: multiple matches for {fname}; using first match: {hits[0]}")
    return hits[0]

file_map = defaultdict(list)
for root, _, files in os.walk(IMAGES_DIR):
    for fname in files:
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            continue
        file_map[fname].append(os.path.abspath(os.path.join(root, fname)))

df_metadata_images['DatasetAbsPath'] = df_metadata_images['DatasetOutFileName'].map(resolve_filename)

missing = df_metadata_images['DatasetAbsPath'].isna().sum()
print(f"Resolved paths for {len(df_metadata_images) - missing}/{len(df_metadata_images)} images. Missing: {missing}")

print(df_metadata_images['weather_desc'].value_counts())
display(df_metadata_images[['DatasetOutFileName', 'DatasetAbsPath']].head())

Resolved paths for 8953/8953 images. Missing: 0
weather_desc
background    5492
sunshine      1586
snow          1033
hail           842
Name: count, dtype: int64


,DatasetOutFileName,DatasetAbsPath
0,000001.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
1,000002.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
2,000003.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
3,000004.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
4,000005.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...


In [4]:
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageFile
import os

# Allow loading truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

class WeatherDataset(Dataset):
    def __init__(self, metadata, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.image_paths = self.metadata['DatasetAbsPath'].tolist() 
        self.labels = self.metadata['weather'].tolist()
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        max_retries = 5
        
        for attempt in range(max_retries):
            try:
                image = Image.open(image_path).convert("RGB")
                
                if self.transform:
                    image = self.transform(image)
                
                label = self.labels[idx]
                return image, label
                
            except Exception as e:
                if attempt == 0:
                    print(f"Error loading {image_path}: {e}")
                # Try a random different sample
                idx = np.random.randint(0, len(self))
                image_path = self.image_paths[idx]
        
        # If all retries fail, raise error
        raise RuntimeError(f"Failed to load any valid image after {max_retries} attempts")


## Utils functions for splits creation

In [ ]:
from typing import Tuple

def is_holdout(df: pd.DataFrame, holdout_col: str = 'holdout') -> pd.Series:
    '''
    Function to determine holdout rows in DataFrame.
    Inputs:
    - df: input DataFrame
    Outputs:
    - pd.Series of booleans indicating holdout status
    '''
    s = df.get('holdout')
    if s is None:
        return pd.Series([False] * len(df), index=df.index)
    if s.dtype == bool:
        return s.fillna(False)
    s_str = s.astype(str).str.upper()
    return s_str == 'TRUE'

def make_splits(df: pd.DataFrame, fold_col: str, fold_index: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''
    Function to split DataFrame into train/val/test sets according to specified fold column and index.
    Inputs:
    - df: input DataFrame
    - fold_col: name of the column containing fold assignments
    - fold_index: index of the fold to use as validation set
    Outputs:
    - train_df: training set DataFrame
    - val_df: validation set DataFrame
    - test_df: test set DataFrame
    '''
    df_cp = df.copy()

    holdout_mask = is_holdout(df_cp)
    test_df = df_cp[holdout_mask].reset_index(drop=True)
    df_no_holdout = df_cp[~holdout_mask].reset_index(drop=True)

    if fold_col not in df_no_holdout.columns:
        raise KeyError(f'Fold column {fold_col} not in DataFrame columns: {list(df_no_holdout.columns)}')

    # Debug: check fold distribution
    # print(f"DEBUG: Fold distribution in non-holdout data:")
    # print(df_no_holdout[fold_col].value_counts().sort_index())
    
    val_mask = df_no_holdout[fold_col] == fold_index
    val_df = df_no_holdout[val_mask].reset_index(drop=True)
    train_df = df_no_holdout[~val_mask].reset_index(drop=True)

    return train_df, val_df, test_df

""" def make_cis_splits(df: pd.DataFrame, fold_index: int = 0):
    return make_splits(df, 'cis_fold', fold_index) """

""" def make_trans_splits(df: pd.DataFrame, fold_index: int = 0):
    return make_splits(df, 'trans_fold', fold_index) """

def make_cis_prod_splits(df: pd.DataFrame, fold_index: int = 0):
    return make_splits(df, 'cis_prod_fold', fold_index)

" def make_trans_splits(df: pd.DataFrame, fold_index: int = 0):\n    return make_splits(df, 'trans_fold', fold_index)\n\ndef make_cis_prod_splits(df: pd.DataFrame, fold_index: int = 0):\n    return make_splits(df, 'cis_prod_fold', fold_index) "

## Visualizing full dataset

In [6]:
import matplotlib.pyplot as plt
import torchvision.transforms as T

dataset = WeatherDataset(metadata=df_metadata_images)

@interact(
    sample_index = widgets.IntSlider(min=0, max=len(dataset)-1, step=1, value=42)
)
def plot_image(sample_index: int) -> None:
    img, label = dataset[sample_index]

    label_str = CLASS_NAMES[label] if 'CLASS_NAMES' in globals() else str(label)
    
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Label: {label_str}")
    plt.axis('off')
    plt.show()


interactive(children=(IntSlider(value=42, description='sample_index', max=8952), Output()), _dom_classes=('wid…

## Training and evaluating DINO models

### DINOv2(w/o registers)

Parlare di augmentations usate, hyperparametri, ecc.

### Custom PyTorch training loop (no Trainer)

Qui definiamo una semplice routine di training/eval usando PyTorch puro e mixed precision (amp).
Il `WeatherDataset` già ritorna dizionari con `pixel_values` e `labels`, quindi il `DataLoader` li impilerà automaticamente.

Vantaggi: controllo più fine, debug più semplice, e spesso migliori opzioni di ottimizzazione per performance.

In [7]:
# Training / evaluation helpers (PyTorch, AMP)
import os
from tqdm.auto import tqdm
import torch.nn as nn

def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(dataloader, desc='train', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        with torch.amp.autocast(enabled=(device.type=='cuda'), device_type=device.type):
            logits = model(images)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': running_loss/total, 'acc': correct/total})
    return running_loss/total, correct/total

def eval_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='eval', leave=False)
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            logits = model(images)
            loss = criterion(logits, labels)
            
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix({'loss': running_loss/total, 'acc': correct/total})
    return running_loss/total, correct/total

def save_checkpoint(model, optimizer, epoch, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, path)


In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import torchvision.transforms as T
import timm

class_names = CLASS_NAMES
num_classes = len(class_names)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 128
# warmup_epochs = 5      # epochs to train only the head
# finetune_epochs = 15   # epochs to train the whole network
# lr_warmup = 5e-3       # higher lr when training only the head
# lr_finetune = 1e-5     # lower lr for full fine-tuning
num_epochs = 20
patience = 5
transfer = False

model_name = 'timm/mobilenetv3_small_100.lamb_in1k'

data_config = timm.data.resolve_data_config({}, model=model_name)

print(f'{model_name} data config:\n {data_config}')

img_size = data_config['input_size'][1]

print(f'Using image size: {img_size}\n')

transforms_train = T.Compose([
    T.Resize(int(img_size / data_config['crop_pct']), interpolation=T.InterpolationMode.BICUBIC), # from model data config
    T.RandomCrop(img_size), # added by me to increase data augmentation
    T.RandomHorizontalFlip(p=0.5), # as in the original paper
    T.RandomVerticalFlip(p=0.5), # as in the original paper
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # added by me to increase data augmentation
    T.ToTensor(), # required
    T.Normalize(mean=data_config['mean'], std=data_config['std']) # required to be uniform with model pre-training(even if we train all the network)
])

transforms_val = T.Compose([
    T.Resize(img_size, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(img_size),
    T.ToTensor(),
    T.Normalize(mean=data_config['mean'], std=data_config['std'])
])

results_mobilenetv3 = []

for fold_idx in range(5):
    print(f'\n{"#"*20} FOLD {fold_idx+1} {"#"*20}')
    train_df, val_df, test_df = make_cis_prod_splits(df_metadata_images, fold_index=fold_idx+1)
    print(f'Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}')

    train_ds = WeatherDataset(train_df, transform=transforms_train)
    val_ds = WeatherDataset(val_df, transform=transforms_val)
    test_ds = WeatherDataset(test_df, transform=transforms_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                              num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, 
                           num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, 
                            num_workers=4, pin_memory=True, persistent_workers=True)
    
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes,
        cache_dir="models-cache"
    )
    if transfer:
        for name, param in model.named_parameters():
            if 'head' not in name:
                param.requires_grad = False

    model.to(device)
    print(f'Model loaded on {device} | Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    scaler = torch.amp.GradScaler(enabled=(device.type=='cuda'))
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.90)

    best_val_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)
        print(f'  Train loss={train_loss:.4f} acc={train_acc:.4f} | Validation loss={val_loss:.4f} acc={val_acc:.4f}')
        scheduler.step()
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            save_checkpoint(model, optimizer, epoch, f'./mobilenetv3_results/fold_{fold_idx}_best.pt')
            print(f'  -> New best validation accuracy: {best_val_acc:.4f}')
        else:
            epochs_without_improvement += 1
            print(f'  -> No improvement for {epochs_without_improvement} epoch(s)')
            
        if epochs_without_improvement >= patience:
            print(f'Early stopping triggered after {epoch+1} epochs')
            break

    test_loss, test_acc = eval_one_epoch(model, test_loader, criterion, device)
    print(f'Fold {fold_idx+1} test loss={test_loss:.4f} acc={test_acc:.4f}')
    results_mobilenetv3.append({'fold': fold_idx+1, 'test_loss': test_loss, 'test_acc': test_acc})

    del model, optimizer, scaler, scheduler, criterion
    torch.cuda.empty_cache()

df_results = pd.DataFrame(results_mobilenetv3)
print('\n===All folds results:')
print(df_results)
print('\nMean:')
print(df_results.mean(numeric_only=True))


Using device: cuda
Data config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.875, 'crop_mode': 'center'}
Using image size: 224x224

==================== FOLD 1 ====================
Train size: 6096, Val size: 1524, Test size: 1333
Loading model...
Model loaded on cuda
Trainable parameters: 1,521,956
Epoch 1/20
Model loaded on cuda
Trainable parameters: 1,521,956
Epoch 1/20


train:   0%|          | 0/96 [00:00<?, ?it/s]

eval:   0%|          | 0/24 [00:00<?, ?it/s]

  Train loss=3.8744 acc=0.5715 | Validation loss=1.5363 acc=0.6371
  -> New best validation accuracy: 0.6371
Epoch 2/20


train:   0%|          | 0/96 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# evaluate the best fold on the test set
